# Vision Assistant — fine-tune YOLO11n on the merged dataset (run `n640_v1`)

**Before running:** Runtime → Change runtime type → **T4 GPU**. Upload `training/data/merged.zip`
(made with `cd training/data && zip -qr merged.zip merged -x "*.cache"`) to Google Drive at `MyDrive/vision/merged.zip`.

Fine-tuning = keep the COCO-pretrained network, swap the output head for our 15 classes, keep training with a
small learning rate. Everything we save goes to `MyDrive/vision/` so a Colab disconnect does not lose the run.

In [ ]:
# Cell 1 — setup: install, mount Drive, unzip the dataset to local disk (much faster than reading from Drive)
!pip -q install ultralytics onnx onnxslim onnxruntime   # onnxslim = graph simplifier used by export(simplify=True)
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/data && mkdir -p /content/data && unzip -q /content/drive/MyDrive/vision/merged.zip -d /content/data
import yaml, json, pathlib
y = yaml.safe_load(open('/content/data/merged/data.yaml'))
y['path'] = '/content/data/merged'          # data.yaml was written on the Mac with an absolute Mac path
yaml.safe_dump(y, open('/content/data/merged/data.yaml', 'w'), sort_keys=False)
classes = y['names']
RUNS = '/content/drive/MyDrive/vision/runs'
print(len(classes), classes)

In [ ]:
# Cell 2 — dataset figure for the report: instances per class per split
import matplotlib.pyplot as plt
counts = json.load(open('/content/data/merged/counts.json'))
plt.figure(figsize=(11, 4))
for i, s in enumerate(['train', 'val', 'test']):
    plt.bar([x + i * 0.27 for x in range(len(classes))], [counts[s].get(c, 0) for c in classes], width=0.27, label=s)
plt.xticks([x + 0.27 for x in range(len(classes))], classes, rotation=60)
plt.yscale('log'); plt.legend(); plt.title('instances per class per split (log scale)'); plt.tight_layout()
plt.savefig('/content/drive/MyDrive/vision/fig_class_counts.png', dpi=150)
plt.show()

In [ ]:
# Cell 3 — train run 1. ~1 h on a T4. Checkpoints go to Drive every epoch (last.pt / best.pt).
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
model.train(data='/content/data/merged/data.yaml', epochs=50, imgsz=640, batch=32, patience=10,
            project=RUNS, name='n640_v1', exist_ok=True, seed=0)

**If Colab disconnected mid-training**, re-run Cell 1 and then this cell instead of Cell 3 — it continues from the
last saved epoch. Otherwise skip it.

In [ ]:
# Cell 3b — resume only if the run was interrupted
from ultralytics import YOLO
YOLO(f'{RUNS}/n640_v1/weights/last.pt').train(resume=True)

In [ ]:
# Cell 4 — evaluate best.pt on the untouched TEST split and save the numbers
from ultralytics import YOLO
best = YOLO(f'{RUNS}/n640_v1/weights/best.pt')
m = best.val(data='/content/data/merged/data.yaml', split='test', imgsz=640, project=RUNS, name='n640_v1_test', exist_ok=True)
per_class = {classes[c]: {'map50': float(ap50), 'map50_95': float(ap)}
             for c, ap50, ap in zip(m.box.ap_class_index, m.box.ap50, m.box.ap)}
print(f'TEST  mAP50 {m.box.map50:.3f}   mAP50-95 {m.box.map:.3f}   (baseline on Mac: 0.496 on 8 COCO classes, 0 on the rest)')
for name, v in per_class.items():
    flag = '  <-- weak, see Task 9' if v['map50'] < 0.2 else ''
    print(f'{name:12s} mAP50 {v["map50"]:.3f}   mAP50-95 {v["map50_95"]:.3f}{flag}')
epochs_trained = sum(1 for _ in open(f'{RUNS}/n640_v1/results.csv')) - 1   # minus header line
json.dump({'run': 'n640_v1', 'epochs_trained': epochs_trained,
           'map50': float(m.box.map50), 'map': float(m.box.map), 'per_class': per_class},
          open(f'{RUNS}/n640_v1_test.json', 'w'), indent=2)

In [ ]:
# Cell 5 — export ONNX for the browser (fixed 1x3x640x640 input, output 1x(4+nc)x8400) and copy to Drive
onnx_path = best.export(format='onnx', opset=17, imgsz=640, simplify=True, dynamic=False)
import shutil
shutil.copy(onnx_path, '/content/drive/MyDrive/vision/best_v1.onnx')
json.dump(classes, open('/content/drive/MyDrive/vision/labels.json', 'w'))
print('saved', onnx_path, '->', 'MyDrive/vision/best_v1.onnx and labels.json')

## Bring home (Task 4 step 4)
From `MyDrive/vision/`:
- `best_v1.onnx` → `public/models/best.onnx`, `labels.json` → `public/models/labels.json`
- `runs/n640_v1/results.csv`, `results.png`, `confusion_matrix.png`, `runs/n640_v1_test.json`, `fig_class_counts.png` → `training/results/n640_v1/`
- File → Download .ipynb → `training/train.ipynb`

---

# Task 9 — second fine-tune and ablations (run `n640_v2` + four arms)

These cells are the Colab equivalent of `training/scripts/run_ablations.py`, which does the same
thing on a local GPU. **The protocol must match whichever machine you use**, otherwise the arms
cannot be compared: 12 epochs, batch 16, seed 0, `close_mosaic` = 20 % of the epoch budget, and the
test split evaluated the same way every time. See `training/results/v2_decisions.md` for why each
of those is fixed.


In [ ]:
# Cell 6 — one helper: train an arm, evaluate it on the untouched TEST split, save its numbers.
# Everything that is NOT the thing being ablated is pinned here so it cannot drift between arms.
import json, time
from ultralytics import YOLO

DATA = '/content/data/merged/data.yaml'
NO_AUG = dict(mosaic=0.0, fliplr=0.0, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, scale=0.0, translate=0.0, erasing=0.0)

def run(name, base='yolo11n.pt', imgsz=640, epochs=12, batch=16, **extra):
    # v1 used close_mosaic=10 of 50 epochs. Keep that ratio: at the default 10, a 12-epoch control
    # would train with mosaic off for 10 of its 12 epochs and the augmentation arm would show nothing.
    close_mosaic = max(1, round(epochs * 0.2))
    t0 = time.time()
    mdl = YOLO(base)
    mdl.train(data=DATA, epochs=epochs, imgsz=imgsz, batch=batch, patience=10, seed=0,
              close_mosaic=close_mosaic, project=RUNS, name=name, exist_ok=True, **extra)
    train_min = (time.time() - t0) / 60

    best = YOLO(f'{RUNS}/{name}/weights/best.pt')
    m = best.val(data=DATA, split='test', imgsz=imgsz, project=RUNS, name=f'{name}_test', exist_ok=True)
    classes = [best.names[i] for i in range(len(best.names))]
    present = list(getattr(m, 'ap_class_index', range(len(classes))))
    out = {'run': name, 'base': base, 'imgsz': imgsz, 'epochs': epochs, 'batch': batch,
           'close_mosaic': close_mosaic, 'augmentation': 'mosaic' not in extra,
           'train_minutes': round(train_min, 1),
           'params': sum(q.numel() for q in best.model.parameters()),
           'map50': float(m.box.map50), 'map50_95': float(m.box.map),
           'per_class_map50': {classes[int(c)]: float(a) for c, a in zip(present, m.box.ap50)},
           'per_class_map50_95': {classes[int(c)]: float(m.box.maps[int(c)]) for c in present},
           'speed_ms': m.speed}
    json.dump(out, open(f'{RUNS}/{name}_test.json', 'w'), indent=2)
    print(name, 'mAP50', round(out['map50'], 3), 'mAP50-95', round(out['map50_95'], 3),
          f"({train_min:.0f} min)")
    return best


In [ ]:
# Cell 7 — the runs. Order matters: the final candidate first, the most expensive arm last, so that
# a Colab disconnect costs you the least important result. Each arm writes its own json, so you can
# stop anywhere and still have whatever finished.

# The final candidate: 30 more epochs from v1's weights, which were still improving at epoch 50.
final = run('n640_v2', base=f'{RUNS}/n640_v1/weights/best.pt', epochs=30, batch=32)

run('n640_e12')                      # control arm for the three ablations below
run('n320_e12', imgsz=320)           # ablation A: image size   (speed vs accuracy)
run('n640_noaug_e12', **NO_AUG)      # ablation C: augmentation (does flip/colour/mosaic help?)
run('s640_e12', base='yolo11s.pt')   # ablation B: model size   (slowest — run it last)


In [ ]:
# Cell 8 — export the final model and check the ONNX file scores the same as the .pt it came from.
# A gap here means the export is broken, not the model.
import shutil
onnx_path = final.export(format='onnx', opset=17, imgsz=640, simplify=True, dynamic=False)
shutil.copy(onnx_path, '/content/drive/MyDrive/vision/best_v2.onnx')

m_onnx = YOLO(onnx_path).val(data=DATA, split='test', imgsz=640)
print('pt  mAP50', round(json.load(open(f'{RUNS}/n640_v2_test.json'))['map50'], 4))
print('onnx mAP50', round(float(m_onnx.box.map50), 4), '  <- should match within rounding noise')


## Bring home (Task 9 step 4)

From `MyDrive/vision/`:
- `best_v2.onnx` → `public/models/best.onnx` (only after it beats v1 on test — see `ablations.md`)
- `runs/*_test.json` → `training/runs/`, and each arm's `results.csv` / `results.png` /
  `confusion_matrix.png` → `training/results/<run>/`

Then, on the Mac: `training/.venv/bin/python training/scripts/quantize.py public/models/best.onnx
public/models/best_int8.onnx` for ablation D, and fill the table in `training/results/ablations.md`.
